# Sanad سند — question answering over Saudi law

## Part 8: "not found" as a threshold

Parts 3, 4 and 6 set how often the model says "not found" through the share of such examples in training (25%, 12%, 18%), and on human questions the behaviour jumped between the settings instead of moving smoothly. Here the trained models are left as they are. For each question the notebook scores how likely the model is to begin with the "not found" sentence, and a threshold on that score, chosen on development questions, decides. If the scores separate answerable from unanswerable questions well, the trade-off can be set after training, without retraining.

**Input:** the outputs of parts 3, 4 and 6 (the adapters). **Settings:** Internet on, GPU accelerator.

Code: https://github.com/readaloush/sanad

In [ ]:
!pip -q install sentence-transformers rank_bm25 peft bitsandbytes accelerate
!git clone -q https://github.com/readaloush/sanad.git
%cd sanad
import glob, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
found = {p.rsplit('/', 2)[-2]: p.rsplit('/', 1)[0] for p in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)}
print(found)
ADAPTERS = ' '.join(['before='] + [f'{k}={v}' for k, v in sorted(found.items())])
print(ADAPTERS)

In [ ]:
!python -m sanad.abstain_curve --adapters {ADAPTERS} --data data --out results/abstain_curve.json 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

In [ ]:
!mkdir -p /kaggle/working/out && cp results/abstain_curve.json /kaggle/working/out/
%cd /kaggle/working
!rm -rf /kaggle/working/sanad